# 📈 Polynomial Regression: A Complete Guide

## 🔍 What is Polynomial Regression?

Polynomial Regression is a form of **regression analysis** in which the relationship between the independent variable $x$ and the dependent variable $y$ is modeled as an **$n$-th degree polynomial**:

$$
y = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + \dots + \beta_n x^n + \varepsilon
$$

Despite the polynomial form, this is still considered **linear regression** because the model is **linear in the coefficients** ($\beta_0, \beta_1, \dots$), even though it's nonlinear in $x$.

> 💡 **Key Insight**: We transform the original feature $x$ into higher-degree features ($x^2, x^3, \dots$) and then apply **linear regression** on these new features.

---

## 🛠️ When to Use Polynomial Regression?

Use polynomial regression when:

- The relationship between features and target is **non-linear** but **smooth** (e.g., curved, parabolic, cubic).
- You observe **curvature** in residual plots from a simple linear model.
- Domain knowledge suggests a polynomial trend (e.g., projectile motion, economic cost curves).
- You need a **simple, interpretable model** that captures non-linearity **without complex algorithms** like neural networks or trees.

> ⚠️ **Not suitable** for:
> - Highly irregular or discontinuous patterns
> - Data with many features (curse of dimensionality)
> - Time-series with seasonality (use Fourier terms or ARIMA instead)

---

## ✅ Pros of Polynomial Regression

| Advantage | Explanation |
|--------|-------------|
| **Simple & Interpretable** | Coefficients have clear meaning; easy to explain to stakeholders. |
| **Captures Non-Linearity** | Can model curves that linear regression cannot. |
| **Built on Linear Regression** | Leverages fast, stable, and well-understood solvers. |
| **No Need for New Algorithms** | Just a feature transformation + linear regression. |

---

## ❌ Cons of Polynomial Regression

| Limitation | Explanation |
|----------|-------------|
| **Overfitting Risk** | High-degree polynomials can fit noise, not signal (especially with small datasets). |
| **Poor Extrapolation** | Polynomials diverge rapidly outside the training range (e.g., $x^{10}$ explodes). |
| **Multicollinearity** | Features like $x$, $x^2$, $x^3$ are highly correlated → unstable coefficient estimates. |
| **Curse of Dimensionality** | In multiple features, polynomial terms grow combinatorially (e.g., $x_1 x_2$, $x_1^2 x_2$, etc.). |

> 📌 **Rule of Thumb**: Start with degree 2 or 3. Only increase if justified by validation performance and domain logic.

---

## 🔬 Practical Tips

- Always **scale features** (e.g., StandardScaler) before generating polynomial terms to reduce numerical instability.
- Use **cross-validation** to choose the best polynomial degree.
- Prefer **regularized variants** (Ridge/Lasso) to control overfitting.
- Visualize **residuals vs. fitted values** to check for remaining patterns.

---

## 🧪 Try It Yourself!

Use the interactive demo in the next code cell to:
- Generate data from a true polynomial
- Fit models of different degrees
- Compare true vs. learned equations
- See overfitting/underfitting in action

In [ ]:
# @title
# Interactive Polynomial Regression with True & Learned Equations
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

plt.style.use('seaborn-v0_8-whitegrid')

# Output areas
controls_output = widgets.Output()
plot_output = widgets.Output()
equation_output = widgets.Output()

# Global vars
X, y, X_train, X_test, y_train, y_test = None, None, None, None, None, None
true_degree = 2
noise_level = 0.1

def format_polynomial(coeffs, var='x'):
    """Convert coefficients to a readable polynomial string."""
    terms = []
    for i, c in enumerate(coeffs):
        if abs(c) < 1e-6:  # treat near-zero as zero
            continue
        if i == 0:
            term = f"{c:.3f}"
        elif i == 1:
            term = f"{c:+.3f}{var}"
        else:
            term = f"{c:+.3f}{var}^{i}"
        terms.append(term)
    if not terms:
        return "0"
    expr = "".join(terms)
    # Fix leading '+' if present
    if expr.startswith('+'):
        expr = expr[1:]
    return expr

def generate_polynomial_data(degree, n_samples=100, noise_std=0.1, seed=None):
    if seed is not None:
        np.random.seed(seed)
    X = np.linspace(-2, 2, n_samples).reshape(-1, 1)
    coeffs = np.random.uniform(-1.5, 1.5, degree + 1)
    if degree == 0:
        coeffs = np.array([np.random.uniform(-1.5, 1.5)])
    y_true = sum(c * (X.ravel() ** i) for i, c in enumerate(coeffs))
    noise = np.random.normal(0, noise_std, n_samples)
    y = y_true + noise
    return X, y, coeffs

def fit_and_plot(X_train, X_test, y_train, y_test, true_coeffs, degrees_to_fit=[1, 2, 3, 4, 5]):
    models = {}
    r2_scores = {}
    X_plot = np.linspace(-2, 2, 400).reshape(-1, 1)

    plt.figure(figsize=(15, 10))

    for i, d in enumerate(degrees_to_fit):
        model = Pipeline([
            ('poly', PolynomialFeatures(degree=d)),
            ('linear', LinearRegression())
        ])
        model.fit(X_train, y_train)
        y_plot = model.predict(X_plot)
        r2 = r2_score(y_test, model.predict(X_test))
        models[d] = model
        r2_scores[d] = r2

        plt.subplot(2, 3, i + 1)
        plt.scatter(X_train, y_train, color='steelblue', alpha=0.7, s=20, label='Train')
        plt.scatter(X_test, y_test, color='crimson', alpha=0.7, s=20, label='Test')
        plt.plot(X_plot, y_plot, color='darkgreen', linewidth=2)
        plt.title(f'Fit Degree {d} | R²: {r2:.3f}', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.5)
        if i == 0:
            plt.legend()

    # Best model
    best_deg = max(r2_scores, key=r2_scores.get)
    best_model = models[best_deg]
    best_pred = best_model.predict(X_plot)

    plt.subplot(2, 3, 6)
    plt.scatter(X_train, y_train, color='steelblue', alpha=0.7, s=20)
    plt.scatter(X_test, y_test, color='crimson', alpha=0.7, s=20)
    plt.plot(X_plot, best_pred, color='purple', linewidth=2.5,
             label=f'Best: Deg {best_deg}\n(R²={r2_scores[best_deg]:.3f})')
    plt.title('Best Fit', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

    # --- Extract learned coefficients ---
    poly_features = best_model.named_steps['poly']
    lin_reg = best_model.named_steps['linear']
    learned_coeffs = lin_reg.coef_.copy()
    learned_coeffs[0] += lin_reg.intercept_  # include intercept in coeff[0]

    # Pad true_coeffs to match display length if needed (for clarity)
    true_eq = format_polynomial(true_coeffs)
    learned_eq = format_polynomial(learned_coeffs)

    # Display equations and scores
    with equation_output:
        clear_output(wait=True)
        display(Markdown(f"### 📌 True Polynomial (Degree {len(true_coeffs)-1})"))
        display(Markdown(f"**y =** `{true_eq}`"))
        display(Markdown(f"### 📌 Learned Polynomial (Best Fit, Degree {best_deg})"))
        display(Markdown(f"**y =** `{learned_eq}`"))

        print("\n" + "="*50)
        print("R² Scores on Test Set")
        print("="*50)
        for d in degrees_to_fit:
            print(f"Fit Degree {d:2d}: {r2_scores[d]:.4f}")
        print(f"\n✅ Best fitted degree: {best_deg}")

# Widgets
degree_dropdown = widgets.Dropdown(
    options=[0, 1, 2, 3, 4, 5],
    value=2,
    description='True Degree:',
    style={'description_width': 'initial'}
)

noise_slider = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Noise Level:',
    readout_format='.2f',
    style={'description_width': 'initial'}
)

generate_button = widgets.Button(
    description="🔄 Generate New Data",
    button_style='primary',
    icon='random'
)

def on_generate_click(b):
    global X, y, X_train, X_test, y_train, y_test, true_degree, noise_level
    true_degree = degree_dropdown.value
    noise_level = noise_slider.value

    with plot_output:
        clear_output(wait=True)
        seed = np.random.randint(0, 100000)
        X, y, true_coeffs = generate_polynomial_data(
            degree=true_degree,
            noise_std=noise_level,
            seed=seed
        )
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )
        fit_and_plot(X_train, X_test, y_train, y_test, true_coeffs)

generate_button.on_click(on_generate_click)

# Layout
with controls_output:
    display(widgets.HBox([degree_dropdown, noise_slider, generate_button]))

# Display UI
display(controls_output)
display(equation_output)   # Equations shown right below controls
display(plot_output)       # Plots below equations

# Initial run
on_generate_click(None)

Output()

Output()

Output()